# 3. Championnat à 16 équipes (Botola)

**Problème.**
- 16 équipes, chacune rencontre toutes les autres deux fois : une fois pendant la phase aller (journées 1 à 15), une fois pendant la phase retour (journées 16 à 30).
- Aucune équipe ne joue plus de 2 matchs de suite à domicile, ni plus de 2 matchs de suite à l'extérieur.

**Taille du problème.** 16 × 15 × 30 = **7 200 variables binaires**, dont 240 valent 1 dans un calendrier (un par match).

**Variable.** $x_{ijk} = 1$ si l'équipe $i$ reçoit l'équipe $j$ lors de la journée $k$.

## Formulation

$$\min\ 0$$

$$\sum_{k=1}^{30} x_{ijk} = 1 \quad \forall i \neq j \qquad \text{(C1)}$$

$$\sum_{j \neq i} \left(x_{ijk} + x_{jik}\right) = 1 \quad \forall i,\ \forall k \qquad \text{(C2 : un match par journée)}$$

$$\sum_{k=1}^{15} \left(x_{ijk} + x_{jik}\right) = 1 \quad \forall i < j \qquad \text{(C3 : phase aller ; le retour se déduit de C1 et C3)}$$

$$\sum_{j \neq i} \sum_{t=k}^{k+2} x_{ijt} \le 2 \quad \forall i,\ k = 1, \dots, 28 \qquad \text{(C4 : pas de DDD)}$$

$$\sum_{j \neq i} \sum_{t=k}^{k+2} x_{jit} \le 2 \quad \forall i,\ k = 1, \dots, 28 \qquad \text{(C5 : pas de EEE)}$$

$$x_{ijk} \in \{0, 1\}$$

### Remarque sur l'énoncé : pourquoi on n'interdit pas « EE »

L'énoncé peut se lire « ni **2** matchs successifs à l'extérieur », c'est-à-dire EE interdit. Cette lecture rend le problème **infaisable** :

1. Chaque équipe joue 15 matchs à l'extérieur sur 30 journées. Si deux E ne peuvent pas se suivre, il n'existe que **16 suites D/E possibles** : l'alternance parfaite EDED…ED ou DEDE…DE, ou une alternance avec un seul DD placé quelque part.
2. Deux équipes qui ont la même suite sont toujours à domicile ou à l'extérieur en même temps : elles ne peuvent jamais se rencontrer. Les 16 équipes doivent donc avoir 16 suites différentes, c'est-à-dire toutes.
3. Parmi ces 16 suites, une seule commence par D. À la journée 1, il y aurait donc 15 équipes à l'extérieur, alors que C2 impose 8 matchs, soit 8 équipes à l'extérieur. Contradiction.

On retient donc la lecture symétrique : pas plus de 2 matchs de suite, ni à domicile, ni à l'extérieur.

In [1]:
import time
import pulp

n = 16
J = 2 * (n - 1)   # 30 journées
A = n - 1         # 15 journées aller

equipes = range(1, n + 1)
journees = range(1, J + 1)

## Modèle

In [2]:
modele = pulp.LpProblem("championnat", pulp.LpMinimize)

x = {(i, j, k): pulp.LpVariable(f"x_{i}_{j}_{k}", cat="Binary")
     for i in equipes for j in equipes if i != j for k in journees}
print("Nombre de variables :", len(x))

modele += 0

# C1
for i in equipes:
    for j in equipes:
        if i != j:
            modele += pulp.lpSum(x[i, j, k] for k in journees) == 1

# C2
for i in equipes:
    for k in journees:
        modele += pulp.lpSum(x[i, j, k] + x[j, i, k] for j in equipes if j != i) == 1

# C3
for i in equipes:
    for j in equipes:
        if i < j:
            modele += pulp.lpSum(x[i, j, k] + x[j, i, k] for k in range(1, A + 1)) == 1

# C4 : pas de DDD
for i in equipes:
    for k in range(1, J - 1):
        modele += pulp.lpSum(x[i, j, t] for j in equipes if j != i
                             for t in (k, k + 1, k + 2)) <= 2

# C5 : pas de EEE
for i in equipes:
    for k in range(1, J - 1):
        modele += pulp.lpSum(x[j, i, t] for j in equipes if j != i
                             for t in (k, k + 1, k + 2)) <= 2

Nombre de variables : 7200


## Recherche de calendriers

On applique la même méthode que pour les petits cas (coupe d'exclusion $\le 239$ après chaque solution). Mais ici, **lister toutes les solutions est hors de portée** : renommer les équipes d'un calendrier valide donne déjà $16! \approx 2 \cdot 10^{13}$ calendriers, et chaque résolution prend une à quelques minutes. On limite donc la recherche à `MAX_SOLUTIONS` calendriers.

In [3]:
MAX_SOLUTIONS = 1
calendriers = []

while len(calendriers) < MAX_SOLUTIONS:
    debut = time.time()
    modele.solve(pulp.PULP_CBC_CMD(msg=0))
    if pulp.LpStatus[modele.status] != "Optimal":
        print("Plus aucune solution :", pulp.LpStatus[modele.status])
        break

    S = [(i, j, k) for (i, j, k) in x if x[i, j, k].value() > 0.5]
    calendriers.append(S)
    print(f"Calendrier {len(calendriers)} trouvé en {time.time() - debut:.0f} s")

    modele += pulp.lpSum(x[v] for v in S) <= len(S) - 1

Calendrier 1 trouvé en 124 s


In [4]:
# Affichage du premier calendrier ("9-2" = l'équipe 9 reçoit l'équipe 2)
S = calendriers[0]
for k in journees:
    print(f"J{k:2d} : " + "  ".join(f"{i}-{j}" for (i, j, kk) in S if kk == k))

J 1 : 1-11  3-6  4-5  7-8  9-2  13-15  14-12  16-10
J 2 : 1-7  2-3  5-6  8-9  10-14  11-15  12-13  16-4
J 3 : 6-10  7-3  9-1  11-8  12-5  13-16  14-2  15-4
J 4 : 1-14  3-16  4-11  5-2  7-12  8-6  10-13  15-9
J 5 : 2-10  3-9  5-7  6-4  8-15  12-11  13-1  16-14
J 6 : 1-12  4-8  6-16  7-13  9-5  11-2  14-3  15-10
J 7 : 2-16  5-11  7-6  9-4  10-8  12-3  13-14  15-1
J 8 : 1-2  3-5  4-10  6-15  8-12  13-9  14-11  16-7
J 9 : 1-4  2-15  5-16  6-13  7-14  8-3  9-12  11-10
J10 : 3-13  4-7  5-8  9-6  10-1  12-2  15-14  16-11
J11 : 1-16  2-7  3-15  6-12  10-5  11-9  13-8  14-4
J12 : 1-3  2-6  7-11  8-14  9-10  13-4  15-5  16-12
J13 : 4-2  5-13  6-1  10-7  11-3  12-15  14-9  16-8
J14 : 1-5  2-8  3-4  6-14  7-9  12-10  13-11  15-16
J15 : 4-12  8-1  9-16  10-3  11-6  13-2  14-5  15-7
J16 : 1-6  2-9  3-7  5-10  8-4  11-13  12-14  16-15
J17 : 2-5  4-3  6-8  7-1  9-14  10-16  13-12  15-11
J18 : 3-10  5-9  8-2  11-4  12-7  14-1  15-6  16-13
J19 : 1-13  3-12  4-14  6-7  8-5  9-15  10-2  11-16
J20 : 2-4  5

## Vérification du calendrier obtenu

In [5]:
def verifier(S):
    # C1 : chaque "i reçoit j" exactement une fois
    assert len(S) == n * (n - 1) and len({(i, j) for (i, j, k) in S}) == n * (n - 1)
    for e in equipes:
        # C2 : un match par journée
        jours = [k for (i, j, k) in S if e in (i, j)]
        assert sorted(jours) == list(journees)
        # C4, C5 : pas 3 matchs de suite au même endroit
        dom = {k: any(i == e and kk == k for (i, j, kk) in S) for k in journees}
        suite = "".join("D" if dom[k] else "E" for k in journees)
        assert "DDD" not in suite and "EEE" not in suite, (e, suite)
    # C3 : toutes les paires se rencontrent pendant l'aller
    paires_aller = {frozenset((i, j)) for (i, j, k) in S if k <= A}
    assert len(paires_aller) == n * (n - 1) // 2
    return "Calendrier valide"

print(verifier(calendriers[0]))

Calendrier valide
